# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display basic metadata
md = dataset.metadata

print("Dataset Title:", md.name)
print("Description:", md.description)
print("Version:", getattr(md, "version", "N/A"))
print("Published:", getattr(md, "datePublished", "N/A"))
print("License:", getattr(md, "license", "N/A"))

## 2. Data Overview
Review available record sets and fields (`@id`s are used for referencing entities).

Let's list all record sets, fields, and columns in the dataset as defined by the schema.

In [ ]:
# List all record sets by @id
record_sets = getattr(md, 'recordSet', [])
if not record_sets:
    print("No record sets defined in schema. Attempting auto-discovery from Croissant schema...")
    # mlcroissant will load inline recordSets via .record_sets()
    record_sets = [rs['@id'] for rs in dataset.record_sets()]

print("Record Sets @id:")
for rs_id in record_sets:
    print(f"- {rs_id}")

print("\nField @id for each record set:")
fields_by_record_set = {}
for rs_id in record_sets:
    rs_obj = dataset.get_record_set(rs_id)
    fields = rs_obj.get('field', []) if rs_obj else []
    if isinstance(fields, dict):
        fields = [fields]
    field_ids = [f['@id'] if isinstance(f, dict) and '@id' in f else str(f) for f in fields]
    fields_by_record_set[rs_id] = field_ids
    print(f"Record set {rs_id}: {field_ids}")

In [ ]:
# Preview a few records from each record set
for rs_id in record_sets:
    print(f"\nSample records from record set {rs_id}:")
    for i, rec in enumerate(dataset.records(record_set=rs_id)):
        if i >= 3:
            break
        print(rec)

## 3. Data Extraction

Load data from all record sets into pandas DataFrames for analysis. Use record set and field `@id`s from the overview above.

In [ ]:
# Load all available record sets into pandas DataFrames
dataframes = {}
for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"\nDataFrame columns for record set {rs_id}: {dataframes[rs_id].columns.tolist()}")
        print(dataframes[rs_id].head())
    else:
        print(f"Record set {rs_id} yielded no records.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes example operations for the primary record set.

We will:
- Select a numeric field (e.g., age at second cancer diagnosis)
- Filter records
- Normalize the numeric field
- Group by a categorical field (e.g., sex or anatomical location)

**Note:** Replace `<numeric_field_id>` and `<group_field>` below with actual column names available in your DataFrame, which should match the field `@id`s.

In [ ]:
# Choose the main record set for EDA
main_rs_id = record_sets[0] if record_sets else None
df = dataframes.get(main_rs_id, pd.DataFrame())
print(f"Main record set chosen for analysis: {main_rs_id}")

# Inspect column names
print("Columns:", df.columns.tolist())

# Try to auto-identify a numeric field and a group field with domain relevance
potential_numeric_fields = [col for col in df.columns if 'age' in col.lower() or 'interval' in col.lower() or df[col].dtype in [int, float]]
numeric_field = potential_numeric_fields[0] if potential_numeric_fields else df.columns[0]
potential_group_fields = [col for col in df.columns if 'sex' in col.lower() or 'location' in col.lower() or 'anatomical' in col.lower() or df[col].dtype == object]
group_field = potential_group_fields[0] if potential_group_fields else None

print(f"Numeric field selected: {numeric_field}")
print(f"Group field selected: {group_field}")

# Filter records for age/interval > threshold (example threshold 50 if field is 'age', otherwise 10)
threshold = 50 if 'age' in numeric_field.lower() else 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by the group field
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below are examples of histogram and boxplot for the chosen numeric field, and a bar plot for group distribution.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not filtered_df.empty:
    plt.figure(figsize=(7,5))
    sns.histplot(filtered_df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    plt.figure(figsize=(7,5))
    sns.boxplot(x=filtered_df[numeric_field])
    plt.title(f"Boxplot of {numeric_field}")
    plt.show()

    if group_field in filtered_df.columns:
        plt.figure(figsize=(7,5))
        group_counts = filtered_df[group_field].value_counts()
        group_counts.plot(kind='bar')
        plt.title(f"Record count by {group_field} (filtered)")
        plt.xlabel(group_field)
        plt.ylabel("Count")
        plt.show()


## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset contains detailed clinicopathological information about cancer survivors with second primary colorectal cancer.
- Data is provided with well-described metadata, fields, and collection protocols, enabling reproducible research.
- You can filter, normalize, and group by key clinical variables using their `@id`.
- The dataset is suitable for downstream modeling, stratification analyses, and exploration of clinical biomarkers.

**Limitations:**
- Small sample size (N=77) and single-center origin may limit generalizability.
- Not recommended for prevalence estimation or risk prediction outside cancer survivor populations.